# 05 · Tools de consulta estilo `jub-agent`

Las tools de `basic.py` (calculadora, notas) sirven para entender el mecanismo, pero
son juguete. Las tools que realmente justifican MCP en un proyecto como `jub-agent` son
las de **consulta de datos**: `list_products`, `get_product`, `search_products(query)`
con una mini-DSL, `generate_plot(...)`, etc. (mira
[`jub-agent/mcp/tools/products.py`](../../jub-agent/mcp/tools/products.py) y
[`search.py`](../../jub-agent/mcp/tools/search.py) si tienes ese repo al lado).

Aquí replicamos ese estilo en [`../mcp_server/tools/catalog.py`](../mcp_server/tools/catalog.py):
un catálogo mock de estaciones/sensores (clima, aire, agua) en
[`../mcp_server/data/mock_catalog.json`](../mcp_server/data/mock_catalog.json), con una
mini-DSL de filtros (`tipo=aire region=Norte valor>50`) — sin depender de la API real
ni de credenciales.

`server.py` ya registra `catalog` junto con `basic`, así que no hace falta tocar nada:
solo arrancamos el servidor de siempre.


In [2]:
import os
import socket
import subprocess
import sys
import time
from pathlib import Path

MCP_SERVER_DIR = Path("..") / "mcp_server"
MCP_PORT = 8102

env = os.environ.copy()
env["MCP_PORT"] = str(MCP_PORT)

server_process = subprocess.Popen(
    [sys.executable, "server.py"],
    cwd=MCP_SERVER_DIR,
    env=env,
)


def esperar_puerto(host, port, timeout=30):
    inicio = time.time()
    while time.time() - inicio < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                return True
        except OSError:
            time.sleep(0.5)
    raise TimeoutError(f"El servidor no abrió el puerto {port} a tiempo")


esperar_puerto("localhost", MCP_PORT)
print(f"Servidor MCP arriba en http://localhost:{MCP_PORT}/mcp")




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                                                              │
│                                FastMCP 3.4.7                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      tutorial-mcp, 3.4.7                         │
│                  🚀 Deplo

Servidor MCP arriba en http://localhost:8102/mcp


## Primero, sin LLM: la mini-DSL a pelo

Antes de meter al agente, vale la pena ver qué le vamos a estar pidiendo que use,
llamando las tools directamente con el cliente MCP crudo.


In [3]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_URL = f"http://localhost:{MCP_PORT}/mcp"


async def explorar_catalogo():
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            r = await session.call_tool("resumen_por_tipo", {})
            print("resumen_por_tipo() ->", r.content[0].text)

            r = await session.call_tool("search_catalogo", {"query": "tipo=aire region=Norte"})
            print("search_catalogo('tipo=aire region=Norte') ->", r.content[0].text)

            r = await session.call_tool("search_catalogo", {"query": "tipo=clima valor>25"})
            print("search_catalogo('tipo=clima valor>25') ->", r.content[0].text)


await explorar_catalogo()


INFO:     127.0.0.1:56280 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:56298 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:56296 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:56304 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:56306 - "POST /mcp HTTP/1.1" 200 OK
resumen_por_tipo() -> [{"tipo":"agua","conteo":2,"promedio":7.05},{"tipo":"aire","conteo":3,"promedio":48.33},{"tipo":"clima","conteo":3,"promedio":24.13}]
INFO:     127.0.0.1:56316 - "POST /mcp HTTP/1.1" 200 OK
search_catalogo('tipo=aire region=Norte') -> [{"id":5,"nombre":"Sensor de Calidad del Aire Norte","tipo":"aire","region":"Norte","valor":62,"unidad":"AQI"}]
INFO:     127.0.0.1:56318 - "POST /mcp HTTP/1.1" 200 OK
search_catalogo('tipo=clima valor>25') -> [{"id":3,"nombre":"Estación Meteorológica Sur","tipo":"clima","region":"Sur","valor":27.1,"unidad":"°C"}]
INFO:     127.0.0.1:56328 - "DELETE /mcp HTTP/1.1" 200 OK




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                                                              │
│                                FastMCP 3.4.7                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                  🖥  Server:      tutorial-mcp, 3.4.7                         │
│                  🚀 Deplo

## Ahora sí, con el agente

El docstring de `search_catalogo` (en `catalog.py`) documenta la mini-DSL con ejemplos
funcionales — igual que hace `jub-agent/mcp/tools/search.py` con su DSL real de
`jub.v1.VS(...).VO(...)`. Esa documentación es lo único que el modelo tiene para
aprender a construir queries válidas: no le explicamos la DSL en las instrucciones del
agente, confiamos en que la lea del docstring de la tool.


In [5]:
from agent_framework import MCPStreamableHTTPTool
from agent_framework.ollama import OllamaChatClient

OLLAMA_URL = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:1.5b"

mcp_tool = MCPStreamableHTTPTool(name="tutorial-mcp", url=MCP_URL)

agent = OllamaChatClient(host=OLLAMA_URL, model=OLLAMA_MODEL).as_agent(
    name="TutorAgent",
    instructions=(
        "Eres un asistente que responde preguntas sobre un catálogo de "
        "estaciones/sensores (clima, aire, agua) usando las herramientas MCP "
        "disponibles. Usa siempre una tool para consultar datos reales; nunca "
        "inventes cifras. Responde en español, breve y directo."
    ),
    tools=mcp_tool,
)

preguntas = [
    "¿Cuántos sensores de aire hay en el catálogo y cuál es su promedio?",
    "Dame las estaciones de clima con temperatura mayor a 25 grados",
    "¿Cuál es el pH promedio de los monitores de agua?",
]

async with agent:
    for pregunta in preguntas:
        resultado = await agent.run(pregunta)
        print(f"P: {pregunta}\nR: {resultado.text}\n")


INFO:     127.0.0.1:37602 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37604 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:37608 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37620 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37626 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37636 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37650 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37662 - "POST /mcp HTTP/1.1" 200 OK
P: ¿Cuántos sensores de aire hay en el catálogo y cuál es su promedio?
R: Tienes 8 sensores de aire en el catálogo. Según el resumen, su promedio de calidad de aire es 48.

INFO:     127.0.0.1:59796 - "POST /mcp HTTP/1.1" 200 OK
P: Dame las estaciones de clima con temperatura mayor a 25 grados
R: La estación meteorológica en la región Sur tiene una temperatura de 27.1°C, que es mayor a 25 grados.

P: ¿Cuál es el pH promedio de los monitores de agua?
R: Lo siento, parece haber habido un error en el proceso de recuperación del resumen por tipo

## De vuelta a `jub-agent`: el mapa completo

Ya viste, de punta a punta, el mismo camino que recorre `jub-agent` en producción.
Esta tabla mapea cada pieza del tutorial a su equivalente real:

| Aquí (tutorial) | Allá (`jub-agent`) |
|---|---|
| `mcp_server/server.py` | `mcp/server.py` |
| `mcp_server/tools/basic.py`, `catalog.py` | `mcp/tools/observatories.py`, `products.py`, `search.py`, ... |
| Mini-DSL de `search_catalogo` sobre JSON mock | DSL real `jub.v1.VS(...).VO(...)` sobre la API de jub |
| `agent/tutor_agent.py::build_agent()` | `agent/jub_agent.py::build_agent()` |
| `agent/main.py` (FastAPI, `/chat`, `/health`) | `agent/main.py` (FastAPI, `/chat` con SSE, sesiones, `/health`) |
| `docker-compose.yml` (ollama + mcp-server + agent) | `docker-compose.yml` (+ chromadb + ui Chainlit) |

Lo que `jub-agent` añade encima de este tutorial, y que puedes explorar directamente en
ese repo como siguiente paso:

- **RAG**: antes de cada respuesta, embebe la pregunta y busca en ChromaDB entidades
  relevantes de jub para inyectarlas como contexto extra (`agent/providers/rag.py`).
- **CAG** (cache-augmented generation): precarga el "schema" de jub como contexto fijo
  (`agent/providers/cag.py`).
- **Historial persistente** en ChromaDB en vez de solo en memoria
  (`agent/providers/history.py`).
- **Selección multi-proveedor de LLM**: NVIDIA NIM → Docker Model Runner → Ollama, en
  cascada según variables de entorno (`agent/jub_agent.py::_build_chat_client`).
- Una **UI real** (Chainlit) hablándole al agente por HTTP/SSE (`ui/app.py`).

**Siguiente:** corre la versión empaquetada en Docker — ver el `README.md` de esta
carpeta, sección "Correr la versión empaquetada (Docker)".


In [6]:
# Limpieza
server_process.terminate()
server_process.wait(timeout=5)
print("Servidor detenido.")


Servidor detenido.
